In [1]:
# ============================================================
# 1. SETUP
# ============================================================

import os
import json
import math
import random
import time

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt   
import seaborn as sns


# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [2]:

DATA_DIR = "/kaggle/input/datasets/prakrutivara/prostructai"  # <-- adjust to your dataset's actual mount path

TRAIN_NPZ = os.path.join(DATA_DIR, "Train_HHblits.npz")
CB513_NPZ = os.path.join(DATA_DIR, "CB513_HHblits.npz")
TS115_NPZ = os.path.join(DATA_DIR, "TS115_HHblits.npz")

ESM_MODEL_NAME = "facebook/esm2_t12_35M_UR50D"  # good accuracy/speed tradeoff for Kaggle GPUs
                                                  # bump to esm2_t30_150M_UR50D if you have quota to spare
MAX_LEN          = 700     # matches the padded length used in this dataset family
N_UNFROZEN_LAYERS = 3   # fine-tune the last N ESM-2 transformer layers, freeze the rest
LSTM_HIDDEN       = 256
LSTM_LAYERS        = 2
DROPOUT            = 0.3
BATCH_SIZE          = 8
GRAD_ACCUM_STEPS    = 3   # effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS
LR                  = 2e-4
ESM_LR              = 2e-5  # lower LR for the pretrained backbone
WEIGHT_DECAY        = 0.01
EPOCHS               = 25
PATIENCE              = 5   # early stopping on val Q3 accuracy
LABEL_SMOOTHING        = 0.05
VAL_FRACTION            = 0.1
OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)


In [3]:

def inspect_npz(path):
    print(f"\n=== {path} ===")
    d = np.load(path, allow_pickle=True)
    for k in d.files:
        arr = d[k]
        print(f"  key='{k}'  shape={arr.shape}  dtype={arr.dtype}")
    return d

for p in [TRAIN_NPZ, CB513_NPZ, TS115_NPZ]:
    inspect_npz(p)



=== /kaggle/input/datasets/prakrutivara/prostructai/Train_HHblits.npz ===
  key='pdbids'  shape=(10848,)  dtype=<U6
  key='data'  shape=(10848, 1632, 68)  dtype=float64

=== /kaggle/input/datasets/prakrutivara/prostructai/CB513_HHblits.npz ===
  key='pdbids'  shape=(513,)  dtype=<U6
  key='data'  shape=(513, 874, 68)  dtype=float64

=== /kaggle/input/datasets/prakrutivara/prostructai/TS115_HHblits.npz ===
  key='pdbids'  shape=(115,)  dtype=<U6
  key='data'  shape=(115, 1111, 68)  dtype=float64


In [4]:
d = np.load(TRAIN_NPZ, mmap_mode="r")
arr = d['data']          # (10848, 1632, 68)
pdbids = d['pdbids']

sample = arr[0]           # one protein: (1632, 68)

# find which rows are real residues vs padding (padding rows are usually all-zero)
row_abs_sum = np.abs(sample).sum(axis=1)
nonzero_rows = np.where(row_abs_sum > 0)[0]
print("protein length guess (nonzero rows):", len(nonzero_rows))
print("first/last nonzero row index:", nonzero_rows[0], nonzero_rows[-1])

valid = sample[nonzero_rows]   # (real_len, 68)

print("\nPer-column stats over the real residues of this one protein:")
print("min :", np.round(valid.min(axis=0), 3))
print("max :", np.round(valid.max(axis=0), 3))
print("mean:", np.round(valid.mean(axis=0), 3))

# check row-sums over plausible one-hot chunks to find category blocks
for start in range(0, 68, 1):
    pass  # (see below, we do it properly)

def block_report(valid, width_guesses=(20, 21, 22, 8, 9, 3, 30, 22)):
    """Scan for contiguous column ranges that look like one-hot blocks
    (values in {0,1}, each row sums to ~1)."""
    n_feat = valid.shape[1]
    for start in range(n_feat):
        for w in width_guesses:
            end = start + w
            if end > n_feat:
                continue
            block = valid[:, start:end]
            is_binary = np.all((np.abs(block) < 1e-6) | (np.abs(block - 1) < 1e-6))
            if not is_binary:
                continue
            row_sums = block.sum(axis=1)
            if np.allclose(row_sums, 1.0, atol=1e-6):
                print(f"  cols [{start}:{end}) width={w} -> looks like a ONE-HOT block (every row sums to 1)")

print("\nScanning for one-hot blocks:")
block_report(valid)

print("\nFirst 3 residues, all 68 raw values (to eyeball manually too):")
for i in range(3):
    print(f"residue {i}:", np.round(valid[i], 3))

protein length guess (nonzero rows): 330
first/last nonzero row index: 0 329

Per-column stats over the real residues of this one protein:
min : [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00
  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00
  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00
  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00
  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00
  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00
  0.000e+00  0.000e+00  0.000e+00  0.000e+00  1.450e-01  0.000e+00
  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00  1.100e-02
  7.400e-02  2.300e-02  1.000e+00  0.000e+00  1.000e+00  0.000e+00
  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00
  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00 -1.775e+02
 -1.776e+02  7.870e+01]
max : [  1.      0.      1.      1.      1.      1.      1.      1.      1.
   1.      1.     

In [5]:
import requests
from difflib import SequenceMatcher

CANDIDATE_AA_ORDERS = {
    "alphabetical":      "ACDEFGHIKLMNPQRSTVWY",
    "hhblits_hhm_order": "ARNDCQEGHILKMFPSTWYV",   # standard PSI-BLAST/HHblits column order
    "cb6133_style":      "ACEDGFIHKMLNQPSRTWVY",
}

sample_idx = 0
aa_onehot = arr[sample_idx, :, 0:20]
real_rows = aa_onehot.sum(axis=1) > 0
aa_argmax = aa_onehot[real_rows].argmax(axis=1)

decoded = {name: "".join(order[i] for i in aa_argmax) for name, order in CANDIDATE_AA_ORDERS.items()}
for name, seq in decoded.items():
    print(f"{name:20s}: {seq[:60]}")

pdb_raw = str(pdbids[sample_idx])
pdb_code = pdb_raw[:4].upper()
print("\nPDB ID:", pdb_raw, "-> fetching", pdb_code)

resp = requests.get(f"https://www.rcsb.org/fasta/entry/{pdb_code}", timeout=15)
print("HTTP status:", resp.status_code)

fasta_seqs, cur = [], []
for line in resp.text.strip().split("\n"):
    if line.startswith(">"):
        if cur: fasta_seqs.append("".join(cur)); cur = []
    else:
        cur.append(line.strip())
if cur: fasta_seqs.append("".join(cur))

print(f"\nReal chains found: {len(fasta_seqs)}")
for i, s in enumerate(fasta_seqs):
    print(f"  chain {i} (len={len(s)}): {s[:60]}")

print("\n--- Similarity of each candidate decoding vs real sequence(s) ---")
for name, seq in decoded.items():
    best = max(SequenceMatcher(None, seq, real).ratio() for real in fasta_seqs)
    print(f"  {name:20s} similarity = {best:.3f}")

alphabetical        : MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVK
hhblits_hhm_order   : LHTAVGAHFPFGSCWHSECSPFIDDPIQIGDWFAMGISPWQNQTFNKISQADHAWFWHWH
cb6133_style        : LKTAYHAKPSPHRGWKRIGRSPMDDSMFMHDWPAQHMRSWFEFTPENMRFADKAWPWKWK

PDB ID: 12as-A -> fetching 12AS
HTTP status: 200

Real chains found: 1
  chain 0 (len=330): MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVK

--- Similarity of each candidate decoding vs real sequence(s) ---
  alphabetical         similarity = 1.000
  hhblits_hhm_order    similarity = 0.009
  cb6133_style         similarity = 0.009


In [6]:

# ==========================
# Dataset definitions
# ==========================


AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")
SS8_ORDER = ["G", "H", "I", "B", "E", "S", "T", "C"]


Q8_TO_Q3 = {
    "G": "H",
    "H": "H",
    "I": "H",
    "B": "E",
    "E": "E",
    "S": "C",
    "T": "C",
    "C": "C",
}

Q3_CLASSES = ["H", "E", "C"]
Q8_CLASSES = SS8_ORDER

def load_split(npz_path):

    d = np.load(npz_path, allow_pickle=True)

    arr = d["data"]
    pdbids = d["pdbids"]

    assert arr.shape[-1] == 68, \
        f"Expected 68 features, got {arr.shape[-1]}"

    aa_block = arr[:, :, 0:20]
    ss_block = arr[:, :, 57:65]

    aa_idx = aa_block.argmax(axis=-1)
    ss_idx = ss_block.argmax(axis=-1)

    sequences = []
    ss8_labels = []

    for i in range(arr.shape[0]):

        # non-padding residues
        real = aa_block[i].sum(axis=-1) > 0

        seq = "".join(
            AA_ORDER[j]
            for j in aa_idx[i][real]
        )

        ss = "".join(
            SS8_ORDER[j]
            for j in ss_idx[i][real]
        )

        sequences.append(seq)
        ss8_labels.append(ss)

    return sequences, ss8_labels, pdbids

print("Loading datasets...")

train_seqs, train_ss8, train_ids = load_split(TRAIN_NPZ)
cb513_seqs, cb513_ss8, cb513_ids = load_split(CB513_NPZ)
ts115_seqs, ts115_ss8, ts115_ids = load_split(TS115_NPZ)

print(f"Train : {len(train_seqs)}")
print(f"CB513 : {len(cb513_seqs)}")
print(f"TS115 : {len(ts115_seqs)}")

Loading datasets...
Train : 10848
CB513 : 513
TS115 : 115


In [7]:
for name, seqs, labels, ids in [

    ("TRAIN", train_seqs, train_ss8, train_ids),
    ("CB513", cb513_seqs, cb513_ss8, cb513_ids),
    ("TS115", ts115_seqs, ts115_ss8, ts115_ids),

]:

    print(f"\n{name}")
    print("PDB :", ids[0])
    print("SEQ :", seqs[0][:80])
    print("SS8 :", labels[0][:80])

    print("Length:", len(seqs[0]))

    assert len(seqs[0]) == len(labels[0])


lengths = [len(x) for x in train_seqs]

print(f"""
Train proteins : {len(lengths)}

Minimum length : {min(lengths)}
Maximum length : {max(lengths)}
Average length : {np.mean(lengths):.2f}
Median length  : {np.median(lengths):.2f}
""")


TRAIN
PDB : 12as-A
SEQ : MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQT
SS8 : CCCCHHHHHHHHHHHHHHHHHHHHHHHCEEECCCCSEEETTSSCSCCTTTTCCCCEECCSSSTTCCEEECSCCTTHHHHH
Length: 330

CB513
PDB : 154l-A
SEQ : RTDCYGNVNRIDTTGASCKTAKPEGLSYCGVSASKKIAERDLQAMDRYKTIIKKVGEKLCVEPAVIAGIISRESHAGKVL
SS8 : CCCTTCCGGGSCCCCBCHHHHTTTTCSCCBHHHHHHHHHHTHHHHHTTHHHHHHHHHHHTSCHHHHHHHHHHHHGGGTTC
Length: 185

TS115
PDB : 5b3d-A
SEQ : TRLSEILDQTTVLNDLKTVDAEQQQLSVGQINGSQLQRITEEKSSLLATLDYLEQQRRLEQNAQRSANDDIAERWQAITE
SS8 : CHHHHHHHHHHHHHHHHHHHHHHHHHCSSCTTCCHHHHHHHHHHHHHHHHHHHHHHHHHTSSSCCCSSCTHHHHHHHHHH
Length: 137

Train proteins : 10848

Minimum length : 11
Maximum length : 1632
Average length : 254.58
Median length  : 217.00



In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL_NAME)

IGNORE_INDEX = -100

Q8_TO_IDX = {c: i for i, c in enumerate(Q8_CLASSES)}


class SSDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


def collate_fn(batch):

    seqs, labels = zip(*batch)

    enc = tokenizer(
        list(seqs),
        padding=True,
        truncation=True,
        max_length=MAX_LEN + 2,
        return_tensors="pt"
    )

    seq_len = enc["input_ids"].shape[1]

    label_tensor = torch.full(
        (len(seqs), seq_len),
        IGNORE_INDEX,
        dtype=torch.long
    )

    for i, lab in enumerate(labels):

        # Number of residue positions available
        max_residues = seq_len - 2

        label_ids = torch.tensor(
            [Q8_TO_IDX[c] for c in lab[:max_residues]],
            dtype=torch.long
        )

        # token 0 = <cls>
        label_tensor[i, 1:1 + len(label_ids)] = label_ids

    return enc, label_tensor


# --------------------------
# Train / Validation split
# --------------------------

lengths = np.array([len(s) for s in train_seqs])

order = np.argsort(lengths)

step = max(1, int(1 / VAL_FRACTION))

val_idx = set(order[::step])

tr_seqs = []
tr_ss8 = []

va_seqs = []
va_ss8 = []

for i, (seq, ss) in enumerate(zip(train_seqs, train_ss8)):

    if i in val_idx:
        va_seqs.append(seq)
        va_ss8.append(ss)
    else:
        tr_seqs.append(seq)
        tr_ss8.append(ss)


train_ds = SSDataset(tr_seqs, tr_ss8)
val_ds   = SSDataset(va_seqs, va_ss8)

cb513_ds = SSDataset(cb513_seqs, cb513_ss8)
ts115_ds = SSDataset(ts115_seqs, ts115_ss8)


train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

cb513_loader = DataLoader(
    cb513_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

ts115_loader = DataLoader(
    ts115_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

print(f"Train: {len(train_ds)}")
print(f"Val:   {len(val_ds)}")
print(f"CB513: {len(cb513_ds)}")
print(f"TS115: {len(ts115_ds)}")

config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Train: 9763
Val:   1085
CB513: 513
TS115: 115


In [9]:
# ==========================================
# ESM-2 + BiLSTM + Classifier
# ==========================================

class ESM_BiLSTM_SSPredictor(nn.Module):

    def __init__(
        self,
        esm_model_name,
        n_classes,
        lstm_hidden=256,
        lstm_layers=2,
        dropout=0.3,
        n_unfrozen_layers=3
    ):

        super().__init__()

        # -----------------------------
        # ESM-2
        # -----------------------------

        self.esm = AutoModel.from_pretrained(
            esm_model_name
        )

        esm_dim = self.esm.config.hidden_size

        # Freeze ALL ESM parameters first
        for p in self.esm.parameters():
            p.requires_grad = False

        # Get transformer layers
        if hasattr(self.esm.encoder, "layer"):
            layers = self.esm.encoder.layer
        else:
            layers = self.esm.encoder.layers

        # Unfreeze last N ESM layers
        for layer in layers[-n_unfrozen_layers:]:
            for p in layer.parameters():
                p.requires_grad = True

        # Unfreeze final ESM LayerNorm
        if hasattr(
            self.esm.encoder,
            "emb_layer_norm_after"
        ):
            for p in self.esm.encoder.emb_layer_norm_after.parameters():
                p.requires_grad = True

        # -----------------------------
        # BiLSTM
        # -----------------------------

        self.lstm = nn.LSTM(
            input_size=esm_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        # Explicitly make sure BiLSTM is TRAINABLE
        for p in self.lstm.parameters():
            p.requires_grad = True

        # -----------------------------
        # Classifier
        # -----------------------------

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            lstm_hidden * 2,
            n_classes
        )

        # Explicitly make classifier TRAINABLE
        for p in self.classifier.parameters():
            p.requires_grad = True


    def forward(
        self,
        input_ids,
        attention_mask
    ):

        # ESM
        esm_out = self.esm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=False,
            return_dict=True
        )

        hidden = esm_out.last_hidden_state

        # Sequence lengths
        lengths = (
            attention_mask
            .long()
            .sum(dim=1)
            .cpu()
        )

        # Pack sequence
        packed = nn.utils.rnn.pack_padded_sequence(
            hidden,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )

        # BiLSTM
        packed_out, _ = self.lstm(packed)

        # Unpack
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out,
            batch_first=True,
            total_length=hidden.shape[1]
        )

        # Classifier
        logits = self.classifier(
            self.dropout(lstm_out)
        )

        return logits

In [10]:
model = ESM_BiLSTM_SSPredictor(
    ESM_MODEL_NAME,
    n_classes=len(Q8_CLASSES),
    lstm_hidden=LSTM_HIDDEN,
    lstm_layers=LSTM_LAYERS,
    dropout=DROPOUT,
    n_unfrozen_layers=N_UNFROZEN_LAYERS
).to(device)

model.safetensors:   0%|          | 0.00/136M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
esm_trainable = sum(
    p.numel()
    for n, p in model.named_parameters()
    if p.requires_grad and n.startswith("esm.")
)

head_trainable = sum(
    p.numel()
    for n, p in model.named_parameters()
    if p.requires_grad and not n.startswith("esm.")
)

total_trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"ESM trainable           : {esm_trainable:,}")
print(f"BiLSTM + classifier     : {head_trainable:,}")
print(f"Total params            : {total_params:,}")
print(f"Total trainable         : {total_trainable:,}")
print(f"Trainable %             : {100 * total_trainable / total_params:.2f}%")

ESM trainable           : 8,314,080
BiLSTM + classifier     : 3,092,488
Total params            : 36,592,889
Total trainable         : 11,406,568
Trainable %             : 31.17%


In [12]:
print("N_UNFROZEN_LAYERS =", N_UNFROZEN_LAYERS)

layers = model.esm.encoder.layer

for i, layer in enumerate(layers):
    trainable = any(p.requires_grad for p in layer.parameters())
    print(f"Layer {i}: {'TRAINABLE' if trainable else 'FROZEN'}")

N_UNFROZEN_LAYERS = 3
Layer 0: FROZEN
Layer 1: FROZEN
Layer 2: FROZEN
Layer 3: FROZEN
Layer 4: FROZEN
Layer 5: FROZEN
Layer 6: FROZEN
Layer 7: FROZEN
Layer 8: FROZEN
Layer 9: TRAINABLE
Layer 10: TRAINABLE
Layer 11: TRAINABLE


In [13]:
esm_total = sum(p.numel() for p in model.esm.parameters())
esm_trainable = sum(p.numel() for p in model.esm.parameters() if p.requires_grad)

head_total = sum(p.numel() for p in model.lstm.parameters()) + \
             sum(p.numel() for p in model.classifier.parameters())

print(f"ESM total params      : {esm_total:,}")
print(f"ESM trainable params  : {esm_trainable:,}")
print(f"BiLSTM + classifier   : {head_total:,}")

ESM total params      : 33,500,401
ESM trainable params  : 8,314,080
BiLSTM + classifier   : 3,092,488


In [14]:
# ==========================================
# Optimizer
# ==========================================

esm_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and n.startswith("esm.")
]

head_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and not n.startswith("esm.")
]

optimizer = torch.optim.AdamW(
    [
        {"params": esm_params, "lr": ESM_LR},
        {"params": head_params, "lr": LR},
    ],
    weight_decay=WEIGHT_DECAY,
)


# ==========================================
# Learning rate scheduler
# ==========================================

steps_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM_STEPS
)

total_steps = steps_per_epoch * EPOCHS

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[ESM_LR, LR],
    total_steps=total_steps,
    pct_start=0.10,
    anneal_strategy="cos",
)


# ==========================================
# Loss
# ==========================================

criterion = nn.CrossEntropyLoss(
    ignore_index=IGNORE_INDEX,
    label_smoothing=LABEL_SMOOTHING,
)


# ==========================================
# Mixed Precision
# ==========================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)


print("Optimizer:", type(optimizer).__name__)
print("Scheduler:", type(scheduler).__name__)
print("Loss: CrossEntropyLoss")
print("Mixed precision:", device.type == "cuda")
print("Total training steps:", total_steps)

Optimizer: AdamW
Scheduler: OneCycleLR
Loss: CrossEntropyLoss
Mixed precision: True
Total training steps: 10175


In [15]:
# ==========================================
# Training / Validation Epoch
# ==========================================

def q8_preds_to_q3(pred_chars):
    return [Q8_TO_Q3[c] for c in pred_chars]


def run_epoch(loader, train=True, accum_steps=1):

    model.train() if train else model.eval()

    total_loss = 0.0
    n_batches = 0

    all_true = []
    all_pred = []

    if train:
        optimizer.zero_grad(set_to_none=True)

    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:

        for step, (enc, labels) in enumerate(loader):

            input_ids = enc["input_ids"].to(
                device, non_blocking=True
            )

            attention_mask = enc["attention_mask"].to(
                device, non_blocking=True
            )

            labels = labels.to(
                device, non_blocking=True
            )

            # -----------------------------
            # Forward pass
            # -----------------------------

            with torch.amp.autocast(
                device_type="cuda",
                enabled=(device.type == "cuda")
            ):

                logits = model(
                    input_ids,
                    attention_mask
                )

                loss = criterion(
                    logits.reshape(-1, logits.shape[-1]),
                    labels.reshape(-1)
                )

                loss_to_backprop = loss / accum_steps

            # -----------------------------
            # Backpropagation
            # -----------------------------

            if train:

                scaler.scale(
                    loss_to_backprop
                ).backward()

                if (
                    (step + 1) % accum_steps == 0
                    or (step + 1) == len(loader)
                ):

                    scaler.unscale_(optimizer)

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=1.0
                    )

                    scaler.step(optimizer)
                    scaler.update()

                    optimizer.zero_grad(set_to_none=True)

                    scheduler.step()

            # -----------------------------
            # Metrics
            # -----------------------------

            total_loss += loss.item()
            n_batches += 1

            preds = logits.argmax(dim=-1)

            mask = labels != IGNORE_INDEX

            all_true.append(
                labels[mask].detach().cpu().numpy()
            )

            all_pred.append(
                preds[mask].detach().cpu().numpy()
            )

    # ==========================================
    # Combine predictions
    # ==========================================

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    # ==========================================
    # Q8 accuracy
    # ==========================================

    q8_acc = accuracy_score(
        y_true,
        y_pred
    )

    # ==========================================
    # Convert Q8 → Q3
    # ==========================================

    true_chars = [
        Q8_CLASSES[i]
        for i in y_true
    ]

    pred_chars = [
        Q8_CLASSES[i]
        for i in y_pred
    ]

    q3_true = [
        Q8_TO_Q3[c]
        for c in true_chars
    ]

    q3_pred = [
        Q8_TO_Q3[c]
        for c in pred_chars
    ]

    q3_acc = accuracy_score(
        q3_true,
        q3_pred
    )

    return (
        total_loss / n_batches,
        q8_acc,
        q3_acc,
        y_true,
        y_pred
    )

In [16]:
# ==========================================
# Training
# ==========================================

history = {
    "train_loss": [],
    "val_loss": [],
    "train_q3": [],
    "val_q3": [],
    "train_q8": [],
    "val_q8": [],
}

best_val_q3 = 0.0
patience_counter = 0

best_ckpt_path = os.path.join(
    OUT_DIR,
    "best_model.pt"
)

print("Starting training...")
print(f"Epochs: {EPOCHS}")
print(f"Patience: {PATIENCE}")
print(f"Best checkpoint: {best_ckpt_path}")

for epoch in range(1, EPOCHS + 1):

    t0 = time.time()

    # -----------------------------
    # Training
    # -----------------------------

    tr_loss, tr_q8, tr_q3, _, _ = run_epoch(
        train_loader,
        train=True,
        accum_steps=GRAD_ACCUM_STEPS
    )

    # -----------------------------
    # Validation
    # -----------------------------

    va_loss, va_q8, va_q3, _, _ = run_epoch(
        val_loader,
        train=False
    )

    # -----------------------------
    # Save history
    # -----------------------------

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)

    history["train_q8"].append(tr_q8)
    history["val_q8"].append(va_q8)

    history["train_q3"].append(tr_q3)
    history["val_q3"].append(va_q3)

    elapsed = time.time() - t0

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"{elapsed:.1f}s | "
        f"Train Loss: {tr_loss:.4f} | "
        f"Train Q3: {tr_q3:.4f} | "
        f"Train Q8: {tr_q8:.4f} | "
        f"Val Loss: {va_loss:.4f} | "
        f"Val Q3: {va_q3:.4f} | "
        f"Val Q8: {va_q8:.4f}"
    )

    # -----------------------------
    # Save BEST model
    # -----------------------------

    if va_q3 > best_val_q3:

        best_val_q3 = va_q3
        patience_counter = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "config": {
                    "esm_model_name": ESM_MODEL_NAME,
                    "lstm_hidden": LSTM_HIDDEN,
                    "lstm_layers": LSTM_LAYERS,
                    "dropout": DROPOUT,
                    "n_unfrozen_layers": N_UNFROZEN_LAYERS,
                    "q8_classes": Q8_CLASSES,
                },
                "epoch": epoch,
                "best_val_q3": best_val_q3,
            },
            best_ckpt_path
        )

        print(
            f"  -> New best Val Q3: {best_val_q3:.4f} "
            f"| checkpoint saved"
        )

    else:

        patience_counter += 1

        print(
            f"  -> No improvement "
            f"({patience_counter}/{PATIENCE})"
        )

        if patience_counter >= PATIENCE:

            print(
                f"\nEarly stopping: no Val Q3 improvement "
                f"for {PATIENCE} epochs."
            )

            break

print("\nTraining complete!")
print(
    f"Best validation Q3 accuracy: "
    f"{best_val_q3:.4f}"
)
print(
    f"Best checkpoint: {best_ckpt_path}"
)

Starting training...
Epochs: 25
Patience: 5
Best checkpoint: /kaggle/working/best_model.pt
Epoch 01/25 | 183.6s | Train Loss: 1.6027 | Train Q3: 0.5275 | Train Q8: 0.4461 | Val Loss: 1.1200 | Val Q3: 0.7870 | Val Q8: 0.6496
  -> New best Val Q3: 0.7870 | checkpoint saved
Epoch 02/25 | 183.4s | Train Loss: 1.0681 | Train Q3: 0.7988 | Train Q8: 0.6711 | Val Loss: 1.0248 | Val Q3: 0.8039 | Val Q8: 0.6859
  -> New best Val Q3: 0.8039 | checkpoint saved
Epoch 03/25 | 185.2s | Train Loss: 1.0136 | Train Q3: 0.8098 | Train Q8: 0.6907 | Val Loss: 0.9987 | Val Q3: 0.8130 | Val Q8: 0.6971
  -> New best Val Q3: 0.8130 | checkpoint saved
Epoch 04/25 | 184.2s | Train Loss: 0.9874 | Train Q3: 0.8159 | Train Q8: 0.7005 | Val Loss: 0.9833 | Val Q3: 0.8168 | Val Q8: 0.7029
  -> New best Val Q3: 0.8168 | checkpoint saved
Epoch 05/25 | 183.4s | Train Loss: 0.9683 | Train Q3: 0.8198 | Train Q8: 0.7071 | Val Loss: 0.9738 | Val Q3: 0.8195 | Val Q8: 0.7069
  -> New best Val Q3: 0.8195 | checkpoint saved
Epoc